# Robinhood Chain Stock Token market observation

## tl;dr

This is a bounded prerequisite feasibility check for one question: **can Stock Token prices observed outside traditional US equity hours contain information about the next traditional-market opening price?**

The notebook inventories Robinhood Stock Tokens, distinguishes reference/oracle prices from genuine market-clearing prices, and identifies the smallest next collection check. It does **not** backtest, define alpha, build a factor, or create a production pipeline.

The asset counts below are a live snapshot from the official endpoint at execution time. The market-data conclusion is intentionally conservative: genuine prices exist at executed secondary-market trades, but reliable historical coverage must be verified venue by venue before the research question is analysis-ready.


## Context & methods

### Key assumptions and boundaries

- Official Robinhood Chain documentation is the primary source.
- `/rhj/assets` is asset metadata, not a history of prices or trades.
- "Overnight" means outside the traditional US regular session; any later study must define the exact timezone/session boundaries and handle holidays and daylight-saving time explicitly.
- A quote or oracle update is not a market-clearing trade. Only an executed secondary-market swap/fill supplies a genuine transaction price.
- This notebook makes one metadata request and a small connectivity check. It does not crawl assets or venues.

Official references (accessed 2026-09-07):

- [Stock Token APIs](https://docs.robinhood.com/chain/stock-token-apis/)
- [Connecting to Robinhood Chain](https://docs.robinhood.com/chain/connecting/)
- [Stock Tokens](https://docs.robinhood.com/chain/stock-tokens/)
- [Building with Stock Tokens](https://docs.robinhood.com/chain/building-with-stock-tokens/)
- [Oracles & Price Feeds](https://docs.robinhood.com/chain/oracles-and-price-feeds/)
- [Token Contracts](https://docs.robinhood.com/chain/contracts/)
- [Lighter Domains](https://docs.robinhood.com/chain/lighter-domains/)
- [Blockscout REST API](https://docs.blockscout.com/devs/apis/rest)
- [Lighter API introduction](https://apidocs.rh.lighter.xyz/docs/get-started)


## Data

### 1. Setup and network identity

Robinhood Chain mainnet and testnet are different networks. Asset counts below use **mainnet chain ID 4663** only; **testnet chain ID 46630** is shown for explicit separation and is never mixed into the mainnet count.


In [1]:
import json
import urllib.request
from datetime import datetime, timezone

import pandas as pd

ASSETS_URL = "https://api.robinhood.com/rhj/assets"
MAINNET_CHAIN_ID = 4663
TESTNET_CHAIN_ID = 46630
MAINNET_RPC = "https://rpc.mainnet.chain.robinhood.com"

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

network_identity = pd.DataFrame([
    {"network": "Robinhood Chain mainnet", "chain_id": MAINNET_CHAIN_ID, "use_here": "asset inventory and real-market reconnaissance"},
    {"network": "Robinhood Chain testnet", "chain_id": TESTNET_CHAIN_ID, "use_here": "excluded; test activity is not market evidence"},
])
display(network_identity)


                   network  chain_id                                        use_here
0  Robinhood Chain mainnet      4663  asset inventory and real-market reconnaissance
1  Robinhood Chain testnet     46630  excluded; test activity is not market evidence


### 2. Fetch one official metadata response

The request uses only Python's standard library plus pandas already used by the project. No cache, retry framework, or asset-by-asset loop is needed for this reconnaissance.


In [2]:
request = urllib.request.Request(
    ASSETS_URL,
    headers={"Accept": "application/json", "User-Agent": "market-observation-notebook/0.1"},
)
with urllib.request.urlopen(request, timeout=20) as response:
    assets_payload = json.load(response)

fetched_at_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")
raw_assets = assets_payload.get("assets", [])
print(f"Fetched at: {fetched_at_utc}")
print(f"HTTP payload type: {type(assets_payload).__name__}")
print(f"Top-level keys: {list(assets_payload)}")
print(f"Assets returned: {len(raw_assets):,}")


Fetched at: 2026-09-07T11:17:30+00:00
HTTP payload type: dict
Top-level keys: ['assets']
Assets returned: 194


### 3. Inspect the raw JSON schema before transforming

The first object is deliberately printed before normalization. This catches schema drift. In the observed 2026-09-07 response, `tradingCapabilities` is nested under `market`, `extended`, and `overnight`; the current API documentation also describes an older/alternate flat shape. The transform below accepts both without guessing that the two shapes are identical.


In [3]:
if not raw_assets:
    raise ValueError("The official endpoint returned no assets; stop rather than report empty counts.")

print("First raw asset (bounded preview):")
print(json.dumps(raw_assets[0], indent=2, ensure_ascii=False))
print("\nObserved asset keys:", sorted(raw_assets[0]))
print("Observed tradingCapabilities keys:", sorted((raw_assets[0].get("tradingCapabilities") or {})))
print("Observed deployment keys:", sorted((raw_assets[0].get("deployments") or [{}])[0]))


First raw asset (bounded preview):
{
  "id": "0x00000000000000000000000000000000022015c295294037bfe416d3e45327b9",
  "tokenSymbol": "CRM",
  "tokenName": "Salesforce • Robinhood Token",
  "deployments": [
    {
      "contractAddress": "0xd95B44124e475743a7589e68F3D74008A5536D44",
      "chainId": 4663,
      "networkName": "Robinhood Chain"
    }
  ],
  "currentMultiplier": "1.000000000000000000",
  "pendingMultiplier": "",
  "status": "ASSET_STATUS_ACTIVE",
  "logoUrl": "https://cdn.robinhood.com/ncw_assets/logos/0xd95b44124e475743a7589e68f3d74008a5536d44.png",
  "tradingCapabilities": {
    "market": {
      "whole": "TRADING_STATUS_TRADABLE",
      "fractional": "TRADING_STATUS_TRADABLE"
    },
    "extended": {
      "whole": "TRADING_STATUS_TRADABLE",
      "fractional": "TRADING_STATUS_TRADABLE"
    },
    "overnight": {
      "whole": "TRADING_STATUS_TRADABLE",
      "fractional": "TRADING_STATUS_TRADABLE"
    }
  },
  "tokenDecimals": 18,
  "isin": "US79466L3024"
}

Observed a

### 4. Build a compact deployment table

There is one row per asset deployment. `canonical_contract_address` comes directly from the official asset registry response. Tradability fields describe the **underlying equity's Robinhood trading capabilities**; they do not prove that the Stock Token itself has continuous DEX liquidity.


In [4]:
rows = []
for asset in raw_assets:
    capabilities = asset.get("tradingCapabilities") or {}
    market = capabilities.get("market") or {}
    extended = capabilities.get("extended") or {}
    overnight = capabilities.get("overnight") or {}
    deployments = asset.get("deployments") or [{}]

    for deployment in deployments:
        rows.append({
            "asset_id": asset.get("id"),
            "token_symbol": asset.get("tokenSymbol"),
            "token_name": asset.get("tokenName"),
            "asset_status": asset.get("status"),
            "chain_id": deployment.get("chainId"),
            "canonical_contract_address": deployment.get("contractAddress"),
            "current_multiplier": asset.get("currentMultiplier"),
            "all_day_tradability": capabilities.get("allDayTradability", overnight.get("whole")),
            "fractional_tradability": capabilities.get("fractionalTradability", market.get("fractional")),
            "extended_hours_fractional_tradability": capabilities.get(
                "extendedHoursFractionalTradability", extended.get("fractional")
            ),
            "all_day_fractional_tradability": overnight.get("fractional"),
        })

token_deployments = pd.DataFrame(rows).sort_values(["token_symbol", "chain_id"], na_position="last")
display(token_deployments.drop(columns="asset_id").head(12))


    token_symbol                                 token_name         asset_status  chain_id                  canonical_contract_address    current_multiplier      all_day_tradability   fractional_tradability extended_hours_fractional_tradability all_day_fractional_tradability
119         AAOI  Applied Optoelectronics • Robinhood Token  ASSET_STATUS_ACTIVE      4663  0x521Cf887E6531c6F667b5BC4D896E5d9bfE8EB2E  1.000000000000000000  TRADING_STATUS_TRADABLE  TRADING_STATUS_TRADABLE               TRADING_STATUS_TRADABLE        TRADING_STATUS_TRADABLE
146         AAPL                    Apple • Robinhood Token  ASSET_STATUS_ACTIVE      4663  0xaF3D76f1834A1d425780943C99Ea8A608f8a93f9  1.000566080061092436  TRADING_STATUS_TRADABLE  TRADING_STATUS_TRADABLE               TRADING_STATUS_TRADABLE        TRADING_STATUS_TRADABLE
156         ABCL      Abcellera Biologics • Robinhood Token  ASSET_STATUS_ACTIVE      4663  0x3139D77Ace0cbAA5bDfD38bD1F1911a794AF0B0e  1.000000000000000000  TRADING_STATUS

## Results

### 5. Required inventory counts

Status, symbol, and all-day counts are asset-level. Chain-ID counts are deployment-level, because one asset may be deployed on more than one chain.


In [5]:
asset_level = token_deployments.drop_duplicates("asset_id")

summary = pd.Series({
    "total_assets_returned": len(raw_assets),
    "active_mainnet_assets": sum(
        asset.get("status") == "ASSET_STATUS_ACTIVE"
        and any(deployment.get("chainId") == MAINNET_CHAIN_ID for deployment in asset.get("deployments", []))
        for asset in raw_assets
    ),
    "unique_token_symbols": pd.Series([asset.get("tokenSymbol") for asset in raw_assets]).nunique(dropna=True),
}, name="count")

display(summary.to_frame())
print("\nStatus distribution (assets)")
display(pd.Series([asset.get("status") for asset in raw_assets]).value_counts(dropna=False).rename("assets").to_frame())
print("\nChain-ID distribution (deployments)")
display(token_deployments["chain_id"].value_counts(dropna=False).rename("deployments").to_frame())
print("\nAll-day-tradability distribution (assets; underlying-equity metadata)")
display(asset_level["all_day_tradability"].value_counts(dropna=False).rename("assets").to_frame())


                       count
total_assets_returned    194
active_mainnet_assets    194
unique_token_symbols     194

Status distribution (assets)
                     assets
ASSET_STATUS_ACTIVE     194

Chain-ID distribution (deployments)
          deployments
chain_id             
4663              194

All-day-tradability distribution (assets; underlying-equity metadata)
                         assets
all_day_tradability            
TRADING_STATUS_TRADABLE     194


### 6. Tiny reasonableness checks

These checks protect the feasibility note from a malformed response or accidental mainnet/testnet mixing; they are not a unit-test suite.


In [6]:
assert len(raw_assets) == summary["total_assets_returned"]
assert summary["unique_token_symbols"] <= summary["total_assets_returned"]
assert token_deployments["chain_id"].notna().all()
assert set(token_deployments.loc[token_deployments["chain_id"] == TESTNET_CHAIN_ID, "chain_id"]) <= {TESTNET_CHAIN_ID}
print("Checks passed: counts reconcile, chain IDs are present, and testnet is not included in active_mainnet_assets.")


Checks passed: counts reconcile, chain IDs are present, and testnet is not included in active_mainnet_assets.


## Price surfaces: do not conflate them

| Surface | What it is | Market-clearing Stock Token price? | Historical collection implication |
|---|---|---:|---|
| Robinhood REST `/rhj/prices/{symbol}` | Raw underlying-equity bid/ask passed through as-is; not multiplier-adjusted | **No** | Useful only as an underlier reference/benchmark. It is not evidence of a Stock Token trade. |
| Per-token Chainlink feed | Onchain oracle equal to underlying share price × token multiplier; official docs say Stock feeds update 24/5 | **No** | Oracle rounds can be read onchain, subject to feed address, heartbeat, staleness, sequencer, and corporate-action pause checks. This is a reference price, not DEX price discovery. |
| RFQ quote | Signed market-maker quote distributed through an aggregator | **Not until executed** | A displayed quote is not a trade. A filled, onchain-settled RFQ is a genuine execution if its fill data can be identified and decoded. |
| AMM / propAMM swap | Executed swap in a Uniswap-style pool or a venue such as Rialto | **Yes** | Swap events/logs yield transaction prices after identifying the exact pool, quote token, decimals, fee, and event ABI. |
| Lighter spot fill | Executed order-book trade on the dedicated Robinhood Chain Lighter instance | **Yes** | The documented API/SDK exposes order-book/trade data and a `/api/v1/candles` endpoint, but spot-market mapping, candle basis, retention, access, and completeness still require a bounded validation. Perpetual or mark-price candles are not acceptable substitutes. |

The Chainlink feed may be onchain and live, but "onchain" does not make it market-clearing. Likewise, a token's underlier being marked all-day tradable does not prove that its DEX market traded overnight.


## Can genuine historical DEX prices be obtained?

### Robinhood Chain RPC and logs

**In principle, yes for AMM/RFQ contracts that settle on Robinhood Chain.** The official public RPC supports standard JSON-RPC/WebSocket access; official docs recommend an archive provider for historical reads and indexing. Once pool/settlement contract addresses and event signatures are known, `eth_getLogs` can retrieve immutable fills/swaps and block timestamps. The public RPC is rate-limited and not recommended for production, but it is adequate for a bounded probe.

The blocker is discovery and interpretation, not EVM access: the reviewed Robinhood docs name Uniswap, Rialto, 0x RFQ, 1inch Fusion, LiFi, and Lighter, but do not provide a complete per-token pool/market registry, quote-token map, ABI/event map, liquidity threshold, or history guarantee.

### Blockscout

**Useful for discovery and spot-checking, not by itself a clean price series.** Robinhood links the mainnet explorer at `robinhoodchain.blockscout.com`. Blockscout's per-instance REST API exposes transactions and token transfers with keyset pagination. A token transfer alone is not a swap or a price; genuine prices require identifying the DEX contract and decoding the venue-specific event/transaction. Explorer indexing can simplify discovery, but completeness and pagination must be reconciled against RPC logs before relying on it.

### Official venues and documented integrations

- Robinhood documents RFQ liquidity, standard AMMs such as Uniswap, Rialto propAMM, and Lighter spot/order-book trading.
- Uniswap/Rialto executed swaps are genuine market-clearing prices if active Stock Token pools and their quote assets can be identified.
- Lighter is a separate instance with its own execution, sequencing, blockspace, liquidity, contract, API, and Lighter app-chain ID (`466324` mainnet). Therefore Robinhood Chain RPC logs should not be assumed to contain every Lighter spot fill. The venue API is the more direct candidate for Lighter history.
- The Lighter SDK documents `/api/v1/candles` with market ID, resolution, start/end timestamps, and count-back. That makes historical collection plausible, but the notebook has not yet verified that a named Stock Token **spot** market exists, that candles are based on executed fills, or that retention and overnight coverage are sufficient.

### Bounded RPC identity probe

This confirms only that the public mainnet endpoint is reachable and reports the documented chain ID. It does not claim that a DEX pool exists.


In [7]:
rpc_request = urllib.request.Request(
    MAINNET_RPC,
    data=json.dumps({"jsonrpc": "2.0", "id": 1, "method": "eth_chainId", "params": []}).encode(),
    headers={"Content-Type": "application/json", "User-Agent": "market-observation-notebook/0.1"},
)
with urllib.request.urlopen(rpc_request, timeout=10) as response:
    rpc_result = json.load(response)

reported_chain_id = int(rpc_result["result"], 16)
print({"rpc": MAINNET_RPC, "reported_chain_id": reported_chain_id, "matches_documented_mainnet": reported_chain_id == MAINNET_CHAIN_ID})
assert reported_chain_id == MAINNET_CHAIN_ID


{'rpc': 'https://rpc.mainnet.chain.robinhood.com', 'reported_chain_id': 4663, 'matches_documented_mainnet': True}


## Feasibility summary

**A. What price surfaces are available?**  
Underlying-equity bid/ask via Robinhood REST; multiplier-adjusted per-token Chainlink oracle prices; executable/executed RFQ, AMM/propAMM, and Lighter spot/order-book venue prices.

**B. Which one is a genuine market-clearing Stock Token price?**  
Only an executed Stock Token secondary-market fill: an AMM/propAMM swap, a filled onchain-settled RFQ, or a Lighter spot fill. REST underlier quotes and Chainlink oracle values are references, not market-clearing token prices.

**C. Can historical overnight token prices be collected?**  
**Plausibly, yes, but not yet as a validated dataset.** AMM/RFQ executions can in principle be decoded from historical RPC logs or Blockscout-assisted discovery; Lighter exposes a candidate historical candle API. Collection still depends on verified market/pool IDs, event semantics, quote assets, timestamp coverage, venue retention, and actual overnight liquidity.

**D. Is the research question currently feasible with reliable data?**  
**Not yet.** The metadata and documented access surfaces establish technical plausibility, but they do not establish a reliable, sufficiently liquid, correctly classified historical series of overnight market-clearing Stock Token trades. A go decision would be premature until one venue/asset passes a coverage and provenance check.

**E. What is the minimum next data-collection step?**  
Choose **one liquid mainnet Stock Token** and **one genuine spot venue**. Resolve the official token contract plus the exact pool/market ID, then collect 1-minute executed-trade candles or raw fills for only 5–10 recent US sessions. Verify timestamps, quote-token units, spot-vs-perpetual classification, gaps, trade counts/volume, and reconcile a few fills to RPC/explorer records. Stop there and review coverage before any backtest, regression, factor, strategy, or paper trading.
